# Topic modelling des chats Telegram avec BERTopic

Ce notebook applique BERTopic au corpus Telegram afin d’explorer les grandes thématiques présentes dans les conversations. Il intervient après les étapes de préparation et de nettoyage du corpus, ainsi qu’après les premiers essais de topic modelling avec LDA.

Contrairement à LDA, qui s’appuie ici sur des textes lemmatisés, BERTopic est appliqué aux messages nettoyés mais non lemmatisés (`text_clean`). L’objectif est de conserver davantage d’information contextuelle et de tirer parti des embeddings multilingues pour regrouper les messages selon leur proximité sémantique.


## 1. Préparation du corpus

Cette première section reconstruit un tableau de travail à partir des exports JSON des chats Telegram. Chaque message est transformé en document individuel pour BERTopic.

Le notebook conserve les métadonnées utiles à l’analyse — fichier d’origine, dossier, nom du chat, identifiant du message, date, auteur et texte nettoyé — afin de pouvoir revenir ensuite aux messages et aux espaces de discussion d’origine.


In [6]:
import pandas as pd
import numpy as np

import spacy

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

from pathlib import Path
import json
import re

import os

In [1]:
import pandas as pd

### 1.1. Chargement des fichiers JSON

Les fichiers JSON sont recherchés récursivement dans le dossier `chats`. Chaque fichier correspond à un espace Telegram exporté. Cette étape permet de vérifier le nombre de fichiers disponibles avant de construire le tableau de messages.


In [7]:
CHATS_DIR = Path("chats")

json_files = sorted(CHATS_DIR.rglob("*.json"))

print("Fichiers JSON trouvés :", len(json_files))
json_files[:5]

Fichiers JSON trouvés : 41


[PosixPath('chats/Crimee/cher_crimee.json'),
 PosixPath('chats/Crimee/refugees_crimee.json'),
 PosixPath('chats/Crimee/refugees_sevastopol.json'),
 PosixPath('chats/Crimee/refugees_simpheropol.json'),
 PosixPath('chats/Krai_de_Krasnodar/assistance_krasnodar.json')]

### 1.2. Extraction et nettoyage minimal du texte


In [8]:
def extract_text(text_field):
    """
    Telegram JSON может хранить text как:
    - строку
    - список строк и словарей
    - пустое значение
    """
    if text_field is None:
        return ""
    
    if isinstance(text_field, str):
        return text_field
    
    if isinstance(text_field, list):
        parts = []
        for item in text_field:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                parts.append(str(item.get("text", "")))
        return "".join(parts)
    
    return str(text_field)



def clean_text_basic(text):
    text = str(text)
    
    # заменяем переносы строк и табы на пробел
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = text.replace("\t", " ")
    
    # убираем лишние пробелы
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

### 1.3. Construction du tableau des messages

Chaque message non vide est ajouté à une liste de lignes, puis transformé en `DataFrame`. Le résultat constitue la base de travail du notebook : une ligne correspond à un message Telegram, avec son texte et ses principales métadonnées.


In [9]:
rows = []

for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    chat_name = data.get("name", file_path.parent.name)
    chat_id = data.get("id", None)
    
    messages = data.get("messages", [])
    
    for msg in messages:
        text = extract_text(msg.get("text"))
        text = clean_text_basic(text)
        
        if text == "":
            continue
        
        rows.append({
            "file": file_path.name,
            "folder": file_path.parent.name,
            "relative_path": str(file_path.relative_to(CHATS_DIR)),
            "chat_name": chat_name,
            "chat_id": chat_id,
            "message_id": msg.get("id"),
            "date": msg.get("date"),
            "from": msg.get("from"),
            "from_id": msg.get("from_id"),
            "type": msg.get("type"),
            "text_clean": text
        })

In [ ]:
df = pd.DataFrame(rows)

df

### 1.4. Définition des documents pour BERTopic

Pour BERTopic, chaque message est traité comme un document. La colonne utilisée est `text_clean`, qui contient le texte brut nettoyé. La liste `texts_chats` sert ensuite à calculer les embeddings et à entraîner les modèles.


In [11]:
TEXT_COL = "text_clean"

texts_chats = df[TEXT_COL].fillna("").astype(str).str.strip().tolist()

print("Nombre de documents pour BERTopic :", len(texts_chats))

Nombre de documents pour BERTopic : 771743


## 2. Stopwords et embeddings

BERTopic regroupe les documents à partir de représentations vectorielles. Cette section prépare donc deux éléments : les stopwords russes, utilisés pour améliorer la représentation lexicale des topics, et les embeddings des messages, utilisés pour la classification sémantique.


### 2.1. Stopwords russes

Les stopwords sont extraits à partir du modèle spaCy russe. Ils seront utilisés dans le `CountVectorizer`, c’est-à-dire au moment où BERTopic construit les mots représentatifs de chaque topic.


In [14]:
nlp = spacy.load("ru_core_news_sm")
stopwords = list(nlp.Defaults.stop_words)

### 2.2. Calcul et sauvegarde des embeddings

Les messages sont encodés avec un modèle `SentenceTransformer` multilingue. Les embeddings sont calculés séparément de BERTopic afin de mieux contrôler la mémoire, la taille des lots et la réutilisation des résultats.

Les embeddings sont sauvegardés dans un fichier `.npy`, ce qui évite de les recalculer à chaque nouvel essai de modèle.


In [20]:
# Лёгкая multilingual-модель.
# Хороший компромисс между качеством, скоростью и памятью.
embedding_model_chats = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="mps"   # если снова будет ошибка памяти, заменить на "cpu"
)

# Для коротких Telegram-сообщений обычно достаточно 128 токенов.
# Это снижает расход памяти и ускоряет обработку.
embedding_model_chats.max_seq_length = 128

# Считаем embeddings отдельно, чтобы BERTopic не делал это внутри.
# Так легче контролировать batch_size и избежать ошибки MPS out of memory.
embeddings_chats_mini = embedding_model_chats.encode(
    texts_chats,
    batch_size=32,              # если памяти не хватит: 16 или 8
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Сохраняем, чтобы не пересчитывать embeddings при каждом запуске BERTopic
np.save("embeddings_chats_minilm.npy", embeddings_chats_mini)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/24117 [00:00<?, ?it/s]

### 2.3. Rechargement et vérification des embeddings

Cette cellule recharge les embeddings sauvegardés et vérifie que leur nombre correspond bien au nombre de messages. Cette vérification est importante, car un décalage entre les textes et les embeddings fausserait l’attribution des topics.


In [11]:
embeddings_chats_mini = np.load("embeddings_chats_minilm.npy")

texts_chats = df["text_clean"].fillna("").astype(str).tolist()

print("Nombre de textes :", len(texts_chats))
print("Shape embeddings :", embeddings_chats_mini.shape)

assert len(texts_chats) == embeddings_chats_mini.shape[0], "Количество текстов и embeddings не совпадает!"

Nombre de textes : 771743
Shape embeddings : (771743, 384)


## 3. Premier modèle BERTopic

Cette section correspond à un premier essai de BERTopic sur les chats Telegram. Elle sert de modèle de référence pour observer le nombre de topics produits, la part de messages classés comme bruit (`topic -1`) et la lisibilité des topics.

Les paramètres combinent UMAP pour la réduction de dimension, HDBSCAN pour la formation des clusters et `CountVectorizer` pour l’extraction des mots représentatifs.


In [16]:
import os

output_dir_chats = "BERTopic/chats"
os.makedirs(output_dir_chats, exist_ok=True)

### 3.1. Entraînement du premier modèle

Le premier modèle utilise des paramètres relativement standards. Les topics obtenus sont ajoutés au tableau des messages, puis les principaux objets de sortie sont sauvegardés : tableau des messages avec topics, informations sur les topics, tableau des topics et modèle BERTopic.


In [17]:
umap_model_chats = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
    low_memory=True
)

hdbscan_model_chats = HDBSCAN(
    min_cluster_size=80,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=False
)

vectorizer_model_chats = CountVectorizer(
    stop_words=stopwords,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.7
)

topic_model_chats = BERTopic(
    embedding_model=None,
    umap_model=umap_model_chats,
    hdbscan_model=hdbscan_model_chats,
    vectorizer_model=vectorizer_model_chats,
    language="multilingual",
    calculate_probabilities=False,
    verbose=True
)

topics_chats, _ = topic_model_chats.fit_transform(
    texts_chats,
    embeddings=embeddings_chats_mini
)

# Добавляем номера тем в датафрейм
df["bertopic_topic"] = topics_chats
df.to_csv(
    os.path.join(output_dir_chats, "messages_with_bertopic_topics_chats.csv"),
    index=False
)

np.save(
    os.path.join(output_dir_chats, "topics_chats.npy"),
    np.array(topics_chats)
)

topic_info_chats.to_csv(
    os.path.join(output_dir_chats, "bertopic_topic_info_chats.csv"),
    index=False
)

topic_model_chats.save(
    os.path.join(output_dir_chats, "bertopic_model_chats"),
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=False
)

2026-04-25 16:39:22,203 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-25 17:38:49,491 - BERTopic - Dimensionality - Completed ✓
2026-04-25 17:38:49,558 - BERTopic - Cluster - Start clustering the reduced embeddings
python(9910) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(9911) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(9912) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(9913) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
2026-04-25 17:40:31,542 - BERTopic - Cluster - Completed ✓
2026-04-25 17:40:31,756 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-25 17:41:11,942 - BERTopic - Representation - Completed ✓
2026-04-25 17:41:32,376 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you

### 3.2. Vérification du nombre de topics et du bruit

BERTopic place dans le topic `-1` les messages qui ne sont pas rattachés de manière suffisamment stable à un cluster. La proportion de bruit est donc un indicateur important pour évaluer si le modèle est exploitable.


In [ ]:
topic_info_chats

In [19]:
# Посмотреть текущее количество тем
topic_info_chats = topic_model_chats.get_topic_info()

n_topics = topic_info_chats[topic_info_chats["Topic"] != -1].shape[0]
n_outliers = topic_info_chats.loc[topic_info_chats["Topic"] == -1, "Count"].iloc[0]

print("Nombre de thèmes sans -1:", n_topics)
print("Nombre de messages dans le bruit:", n_outliers)
print("Proportion de bruit:", round(n_outliers / len(df), 3))

Nombre de thèmes sans -1: 923
Nombre de messages dans le bruit: 393143
Proportion de bruit: 0.509


### 3.3. Export des messages classés comme bruit

Les messages associés au topic `-1` sont sauvegardés séparément. Cela permet d’examiner qualitativement ce que le modèle n’a pas réussi à regrouper : messages très courts, demandes isolées, formulations ambiguës ou contenus trop hétérogènes.


In [ ]:
import os

# Создаём папку для результатов BERTopic по чатам
output_dir_chats = "BERTopic/chats"
os.makedirs(output_dir_chats, exist_ok=True)

# Добавляем темы в датафрейм, если ещё не добавлены
df["bertopic_topic"] = topics_chats

# Отбираем только сообщения, которые BERTopic отправил в шум
df_noise_chats = df[df["bertopic_topic"] == -1].copy()

# Сохраняем шумовые сообщения отдельно
df_noise_chats.to_csv(
    os.path.join(output_dir_chats, "messages_bertopic_noise_minus1_chats.csv"),
    index=False
)

# Быстрая проверка
print("Total messages : ", len(df))
print("Messages in topic -1:", len(df_noise_chats))
print("Noise ratio:", round(len(df_noise_chats) / len(df), 3))

df_noise_chats.head(20)

## 4. Deuxième modèle BERTopic : réduction du bruit

Le deuxième essai vise à obtenir moins de messages classés comme bruit. Les paramètres sont rendus plus souples : HDBSCAN accepte des clusters plus petits et moins stricts, tandis que UMAP produit une structure légèrement plus lissée.

Cette version cherche donc un compromis entre la conservation de topics suffisamment fins et la réduction du nombre de messages exclus dans `-1`.


In [1]:
import os
import numpy as np
import pandas as pd

from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer


In [30]:
import numpy as np

embeddings_chats_mini = np.load(
    "embeddings_chats_minilm.npy"
)

print("Shape embeddings:", embeddings_chats_mini.shape)

Shape embeddings: (771743, 384)


In [31]:
print("Nombre de textes :", len(texts_chats))
print("Shape embeddings :", embeddings_chats_mini.shape)

assert len(texts_chats) == embeddings_chats_mini.shape[0]

Nombre de textes : 771743
Shape embeddings : (771743, 384)


### 4.1. Entraînement et sauvegarde du modèle plus souple

Le modèle `soft_model` est entraîné sur les mêmes textes et embeddings que le premier modèle. Les résultats sont sauvegardés dans un dossier séparé afin de pouvoir comparer les versions sans écraser les sorties précédentes.


In [ ]:
# Папки для новой модели
BASE_DIR = "BERTopic/chats/soft_model"
os.makedirs(f"{BASE_DIR}/tables", exist_ok=True)
os.makedirs(f"{BASE_DIR}/model", exist_ok=True)


# UMAP: чуть больше соседей = более сглаженная структура,
# меньше микрокластеров и потенциально меньше шума
umap_model_chats_soft = UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
    low_memory=True
)


# HDBSCAN: более мягкая кластеризация
# min_cluster_size=50 вместо 80 позволяет не выкидывать мелкие, но реальные темы
# min_samples=3 вместо 10 делает модель менее строгой → меньше -1
hdbscan_model_chats_soft = HDBSCAN(
    min_cluster_size=50,
    min_samples=3,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=False
)


# Vectorizer только описывает темы словами
vectorizer_model_chats_soft = CountVectorizer(
    stop_words=stopwords,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.7
)


topic_model_chats_soft = BERTopic(
    embedding_model=None,
    umap_model=umap_model_chats_soft,
    hdbscan_model=hdbscan_model_chats_soft,
    vectorizer_model=vectorizer_model_chats_soft,
    language="multilingual",
    calculate_probabilities=False,
    verbose=True
)


topics_chats_soft, _ = topic_model_chats_soft.fit_transform(
    texts_chats,
    embeddings=embeddings_chats_mini
)


# Добавляем темы в df
df["bertopic_topic_soft"] = topics_chats_soft


# Сохраняем сообщения с новыми темами
df.to_csv(
    f"{BASE_DIR}/tables/messages_with_bertopic_topics_chats_soft.csv",
    index=False
)


# Сохраняем отдельно массив тем
np.save(
    f"{BASE_DIR}/tables/topics_chats_soft.npy",
    np.array(topics_chats_soft)
)


# Сохраняем topic_info
topic_info_chats_soft = topic_model_chats_soft.get_topic_info()

topic_info_chats_soft.to_csv(
    f"{BASE_DIR}/tables/topic_info_chats_soft.csv",
    index=False
)


# Сохраняем отдельно шум -1
df_noise_soft = df[df["bertopic_topic_soft"] == -1]

df_noise_soft.to_csv(
    f"{BASE_DIR}/tables/messages_noise_topic_minus1_chats_soft.csv",
    index=False
)


# Сохраняем модель BERTopic
topic_model_chats_soft.save(
    f"{BASE_DIR}/model/bertopic_model_chats_soft",
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=False
)


# Проверка
print("Nombre de documents :", len(texts_chats))
print("Nombre de messages en bruit (-1) :", len(df_noise_soft))
print("Part du bruit :", round(len(df_noise_soft) / len(texts_chats), 4))
print("Nombre de topics hors bruit :", topic_info_chats_soft[topic_info_chats_soft["Topic"] != -1].shape[0])

topic_info_chats_soft.head(30)

## 5. Troisième modèle BERTopic : topics plus généraux

Le troisième essai cherche à produire des topics plus larges et plus interprétables. Les paramètres sont ajustés pour favoriser des regroupements plus généraux : davantage de voisins dans UMAP, taille minimale de cluster plus élevée et filtrage plus strict des mots rares.

Cette version permet de tester si une granularité plus large est plus adaptée à l’analyse qualitative des discussions Telegram.


In [ ]:

BASE_DIR = "BERTopic/chats/general_model"

os.makedirs(f"{BASE_DIR}/tables", exist_ok=True)
os.makedirs(f"{BASE_DIR}/figures", exist_ok=True)
os.makedirs(f"{BASE_DIR}/model", exist_ok=True)


# UMAP: более глобальная структура корпуса
umap_model_chats_general = UMAP(
    n_neighbors=50,        # больше соседей → темы становятся более общими
    n_components=5,
    min_dist=0.1,          # меньше сжатия в микрокластеры
    metric="cosine",
    random_state=42,
    low_memory=True
)


# HDBSCAN: укрупняем темы
hdbscan_model_chats_general = HDBSCAN(
    min_cluster_size=300,  # тема должна быть достаточно крупной
    min_samples=5,         # мягко, чтобы не раздувать шум
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=False
)


# Vectorizer: описание тем словами
vectorizer_model_chats_general = CountVectorizer(
    stop_words=stopwords,
    ngram_range=(1, 2),
    min_df=20,             # убирает редкие слова/выражения
    max_df=0.7
)


topic_model_chats_general = BERTopic(
    embedding_model=None,
    umap_model=umap_model_chats_general,
    hdbscan_model=hdbscan_model_chats_general,
    vectorizer_model=vectorizer_model_chats_general,
    language="multilingual",
    calculate_probabilities=False,
    verbose=True
)


topics_chats_general, _ = topic_model_chats_general.fit_transform(
    texts_chats,
    embeddings=embeddings_chats_mini
)


# ============================================================
# Сохранение результатов
# ============================================================

df["bertopic_topic_general"] = topics_chats_general

df.to_csv(
    f"{BASE_DIR}/tables/messages_with_bertopic_topics_chats_general.csv",
    index=False
)

np.save(
    f"{BASE_DIR}/tables/topics_chats_general.npy",
    np.array(topics_chats_general)
)

topic_info_chats_general = topic_model_chats_general.get_topic_info()

topic_info_chats_general.to_csv(
    f"{BASE_DIR}/tables/topic_info_chats_general.csv",
    index=False
)

df_noise_general = df[df["bertopic_topic_general"] == -1]

df_noise_general.to_csv(
    f"{BASE_DIR}/tables/messages_noise_topic_minus1_chats_general.csv",
    index=False
)


topic_model_chats_general.save(
    f"{BASE_DIR}/model/bertopic_model_chats_general",
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=False
)


# ============================================================
# Визуализации
# ============================================================

fig_barchart_general = topic_model_chats_general.visualize_barchart(
    top_n_topics=30,
    n_words=8,
    title="Top 30 BERTopic topics in chats: general model"
)

fig_barchart_general.write_html(
    f"{BASE_DIR}/figures/barchart_topics_chats_general.html"
)


fig_map_general = topic_model_chats_general.visualize_topics(
    title="Intertopic distance map: BERTopic chats general model"
)

fig_map_general.write_html(
    f"{BASE_DIR}/figures/intertopic_map_chats_general.html"
)


# ============================================================
# Проверка
# ============================================================

print("Nombre de documents :", len(texts_chats))
print("Nombre de messages en bruit (-1) :", len(df_noise_general))
print("Part du bruit :", round(len(df_noise_general) / len(texts_chats), 4))
print(
    "Nombre de topics hors bruit :",
    topic_info_chats_general[topic_info_chats_general["Topic"] != -1].shape[0]
)

topic_info_chats_general.head(30)

## 6. Benchmark de réduction des topics

Après les premiers essais, le notebook teste plusieurs réductions du nombre de topics à partir du modèle retenu. La fonction `reduce_topics` ne réduit pas le bruit `-1` : elle sert seulement à regrouper les topics déjà identifiés.

La stratégie consiste donc à partir d’un modèle jugé exploitable, puis à comparer plusieurs niveaux de réduction — ici 40, 50, 60, 70, 80 et 100 topics — afin de choisir le niveau de granularité le plus lisible pour l’analyse.


In [20]:
from bertopic import BERTopic

MODEL_PATH = "BERTopic/chats/bertopic_model_chats"

topic_model_chats = BERTopic.load(MODEL_PATH)

2026-04-26 13:57:47,413 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


In [17]:
import os
import copy
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [23]:
# Папка первой сохранённой модели
BASE_DIR = "BERTopic/chats"

REDUCE_DIR = "BERTopic/chats/first_model_reduced_compare"

os.makedirs(f"{REDUCE_DIR}/tables", exist_ok=True)
os.makedirs(f"{REDUCE_DIR}/figures", exist_ok=True)
os.makedirs(f"{REDUCE_DIR}/models", exist_ok=True)

In [19]:
topic_counts_to_test = [40, 50, 60, 70, 80, 100]

### 6.1. Comparaison des configurations réduites

Pour chaque valeur testée, le notebook sauvegarde : la liste des topics réduits, les messages associés à ces topics, les messages classés comme bruit, des indicateurs de comparaison et des visualisations HTML.

Le benchmark permet de comparer la taille des topics, leur nombre réel, la part de bruit et la lisibilité des mots-clés associés à chaque configuration.


In [24]:
# Варианты укрупнения тем
benchmark_rows = []

for nr in tqdm(topic_counts_to_test, desc="Benchmark reduce_topics for first model"):
    print(f"\nRéduction vers nr_topics={nr}")

    # Копируем первую обученную модель
    model_reduced = copy.deepcopy(topic_model_chats)

    # Укрупняем темы
    model_reduced.reduce_topics(
        texts_chats,
        nr_topics=nr
    )

    # Таблица тем после редукции
    topic_info_reduced = model_reduced.get_topic_info()

    topic_info_reduced.to_csv(
        f"{REDUCE_DIR}/tables/topic_info_reduced_{nr}.csv",
        index=False
    )

    # Новые темы для каждого сообщения
    reduced_topics = model_reduced.topics_

    np.save(
        f"{REDUCE_DIR}/tables/topics_reduced_{nr}.npy",
        np.array(reduced_topics)
    )

    # Сообщения + новая колонка с reduced topics
    df_reduced = df.copy()
    df_reduced[f"bertopic_topic_reduced_{nr}"] = reduced_topics

    df_reduced.to_csv(
        f"{REDUCE_DIR}/tables/messages_with_topics_reduced_{nr}.csv",
        index=False
    )

    # Отдельно сохраняем шум
    df_noise_reduced = df_reduced[
        df_reduced[f"bertopic_topic_reduced_{nr}"] == -1
    ]

    df_noise_reduced.to_csv(
        f"{REDUCE_DIR}/tables/messages_noise_reduced_{nr}.csv",
        index=False
    )

    # Метрики
    noise_count = int((np.array(reduced_topics) == -1).sum())
    noise_share = noise_count / len(texts_chats)

    topics_without_noise = topic_info_reduced[
        topic_info_reduced["Topic"] != -1
    ]

    benchmark_rows.append({
        "nr_topics_requested": nr,
        "n_topics_real_without_noise": topics_without_noise.shape[0],
        "noise_count": noise_count,
        "noise_share": round(noise_share, 4),
        "median_topic_size": topics_without_noise["Count"].median(),
        "min_topic_size": topics_without_noise["Count"].min(),
        "max_topic_size": topics_without_noise["Count"].max()
    })

    # Bar chart
    fig_barchart = model_reduced.visualize_barchart(
        top_n_topics=nr,
        n_words=8,
        title=f"All reduced BERTopic topics in chats, nr_topics={nr}"
    )

    fig_barchart.write_html(
        f"{REDUCE_DIR}/figures/barchart_reduced_{nr}.html"
    )

    # Intertopic map
    fig_map = model_reduced.visualize_topics(
        title=f"Intertopic distance map for reduced BERTopic topics in chats, nr_topics={nr}"
    )

    fig_map.write_html(
        f"{REDUCE_DIR}/figures/intertopic_map_reduced_{nr}.html"
    )

    # Сохраняем reduced model
    model_reduced.save(
        f"{REDUCE_DIR}/models/bertopic_model_chats_reduced_{nr}",
        serialization="safetensors",
        save_ctfidf=True,
        save_embedding_model=False
    )

    print(f"Saved outputs for nr_topics={nr}")


benchmark_df = pd.DataFrame(benchmark_rows)

benchmark_df.to_csv(
    f"{REDUCE_DIR}/tables/reduction_benchmark.csv",
    index=False
)

benchmark_df

Benchmark reduce_topics for first model:   0%|          | 0/6 [00:00<?, ?it/s]


Réduction vers nr_topics=40


2026-04-26 14:09:43,788 - BERTopic - Topic reduction - Reducing number of topics
2026-04-26 14:09:44,932 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-26 14:10:18,638 - BERTopic - Representation - Completed ✓
2026-04-26 14:10:18,774 - BERTopic - Topic reduction - Reduced number of topics from 924 to 40
2026-04-26 14:10:37,390 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


Saved outputs for nr_topics=40

Réduction vers nr_topics=50


2026-04-26 14:10:38,910 - BERTopic - Topic reduction - Reducing number of topics
2026-04-26 14:10:39,871 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-26 14:11:10,405 - BERTopic - Representation - Completed ✓
2026-04-26 14:11:10,529 - BERTopic - Topic reduction - Reduced number of topics from 924 to 50
2026-04-26 14:11:27,793 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


...

2026-04-26 14:13:10,476 - BERTopic - Topic reduction - Reducing number of topics
2026-04-26 14:13:11,426 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-26 14:13:42,075 - BERTopic - Representation - Completed ✓
2026-04-26 14:13:42,197 - BERTopic - Topic reduction - Reduced number of topics from 924 to 80
2026-04-26 14:14:00,061 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


Saved outputs for nr_topics=80

Réduction vers nr_topics=100


2026-04-26 14:14:01,659 - BERTopic - Topic reduction - Reducing number of topics
2026-04-26 14:14:02,626 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-26 14:14:33,299 - BERTopic - Representation - Completed ✓
2026-04-26 14:14:33,421 - BERTopic - Topic reduction - Reduced number of topics from 924 to 100
2026-04-26 14:14:51,564 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


Saved outputs for nr_topics=100


,nr_topics_requested,n_topics_real_without_noise,noise_count,noise_share,median_topic_size,min_topic_size,max_topic_size
0,40,39,393143,0.5094,2277.0,103,89794
1,50,49,393143,0.5094,2086.0,103,63700
2,60,59,393143,0.5094,1723.0,103,49891
3,70,69,393143,0.5094,1569.0,102,49210
4,80,79,393143,0.5094,1620.0,102,33780
5,100,99,393143,0.5094,1396.0,94,33780


## 7. Choix d’une configuration à 80 topics

À partir du benchmark, la configuration à 80 topics est retenue comme compromis entre granularité et lisibilité. Elle permet de conserver des thèmes suffisamment différenciés tout en évitant une fragmentation excessive du corpus.


### 7.1. Inspection interactive des topics

Cette cellule charge la table `topic_info_reduced_80.csv` et crée une interface de lecture paginée. Elle permet d’examiner les topics par groupes de lignes, en observant leur taille et leurs mots représentatifs.


In [2]:
import math
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Чтобы pandas не обрезал длинный текст
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

path = "/Users/quentinnippert/Documents/mm_files/Telegram_analyse/BERTopic/chats/first_model_reduced_compare/tables/topic_info_reduced_80.csv"
topic_info_reduced_80 = pd.read_csv(path)

df = topic_info_reduced_80
page_size = 20
total_pages = math.ceil(len(df) / page_size)

page = widgets.IntSlider(
    value=1, min=1, max=max(1, total_pages), step=1,
    description="Page", continuous_update=False
)

out = widgets.Output()

def show_page(change=None):
    p = page.value
    start = (p - 1) * page_size
    end = start + page_size
    chunk = df.iloc[start:end].copy()

    with out:
        clear_output(wait=True)
        html_table = chunk.to_html(index=False, escape=False)
        display(HTML(f"""
        <style>
            .wide-table-wrap {{
                max-height: 650px;
                overflow: auto;
                border: 1px solid #ddd;
                padding: 6px;
            }}
            .wide-table-wrap table {{
                border-collapse: collapse;
                width: max-content;
                min-width: 100%;
            }}
            .wide-table-wrap th, .wide-table-wrap td {{
                white-space: pre-wrap;
                word-break: break-word;
                max-width: 1000px;
                vertical-align: top;
                padding: 6px 8px;
            }}
        </style>
        <div class='wide-table-wrap'>{html_table}</div>
        """))
        print(f"Rows {start+1}-{min(end, len(df))} of {len(df)} | Page {p}/{total_pages}")

page.observe(show_page, names="value")
show_page()
display(page, out)

IntSlider(value=1, continuous_update=False, description='Page', max=4, min=1)

Output()

### 7.2. Retour qualitatif aux messages

Après l’examen des mots-clés, le notebook revient aux messages associés à un topic donné. Cette étape est essentielle pour interpréter les topics : elle permet de vérifier ce que les messages disent réellement, d’identifier les contextes de discussion et d’éviter une interprétation fondée uniquement sur les mots représentatifs.


In [ ]:
import pandas as pd

messages_with_topics = pd.read_csv(
    "/Users/quentinnippert/Documents/mm_files/Telegram_analyse/BERTopic/chats/first_model_reduced_compare/tables/messages_with_topics_reduced_80.csv"
)

messages_with_topics.head()

In [ ]:
bertopic_topic_reduced_80 = 39
# file = "refugies_en_russie_RU.json"

# Показывать полный текст в колонках
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_columns", None)

subset = messages_with_topics[
    (messages_with_topics["bertopic_topic_reduced_80"] == bertopic_topic_reduced_80)
    # & (messages_with_topics["file"] == file)
][["text_clean", "file", "date"]].sort_values(by="date", ascending=True)

display(subset.head(60))